# z620 - LightGBM tweedie por cluster (Etapa D, consigna nueva)
`objective=tweedie`, `max_bin=1023`, Optuna, SOLO campos escalados como features. Predict en espacio escalado -> desescalar por `TN_promedio` de cada fila. Sin walk-forward (segun consigna, no vale la pena).

**ADVERTENCIA TECNICA (no resuelta, se deja explicita):** `objective=tweedie` de LightGBM requiere target &ge;0. El target `clase` es un DELTA (`clase_original_escalada - tn0_escalado`) y puede ser NEGATIVO cuando las ventas caen entre el ancla y el periodo objetivo. Esto es una contradiccion entre dos partes de la consigna (target=delta con signo, objetivo=tweedie que exige no-negatividad). Se implementa literal como se pidio; si LightGBM tira error o entrena mal por esto, hay que decidir explicitamente como resolverlo (no se decide aca por cuenta propia).

In [3]:
!pip install -q lightgbm pyarrow optuna polars

In [4]:
import os
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import optuna
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [5]:
PARAM = {
    'experimento': 'LGB12_ESCALADO_CLUSTER',
    'kaggle_competition': 'labo-iii-2026-ba',
    'features_path': '/home/ds/exp/CP_CLUSTER_BRUTO/tb_FE_CP_con_cluster.parquet',
    'apredecir_path': '/home/ds/datasets/product_id_apredecir201912.txt',
    'periodo_ultimo_dato': 201912,
    'semilla': 102103,
    'n_trials': 30,          # por cluster -- bajar si son muchos clusters y el tiempo total importa
    'max_bin': 1023,
    'min_filas_por_cluster': 500
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/LGB12_ESCALADO_CLUSTER


In [6]:
df = pl.read_parquet(PARAM['features_path'])
print(df.shape)
print("minimo de 'clase' (chequeo de negatividad para tweedie):", df["clase"].min())

(16648066, 67)
minimo de 'clase' (chequeo de negatividad para tweedie): -33.99948411603997


## Definicion de features: SOLO campos escalados + categoricas de jerarquia
Se excluyen explicitamente los campos en magnitud real (`tn`, `tn0`, `tn1`, `TN_promedio`) y los que arman el target (`clase_original`, `clase_original_escalada`, `clase`).

In [5]:
cols_excluir = {
    "tn", "tn0", "tn1", "TN_promedio",
    "clase_original", "clase_original_escalada", "clase",
    "periodo", "periodo_target_m", "E_tn_shift1", "cluster_id"
}
features = [c for c in df.columns if c not in cols_excluir]
categoricas = [c for c in ["customer_id", "product_id", "cat1", "cat2", "cat3", "brand", "descripcion"] if c in features]
print(len(features), "features")
print(features)

56 features
['customer_id', 'product_id', 'tn0_escalado', 'E_tn', 'periodo_m', 'E_tn_lag_1', 'E_tn_lag_2', 'E_tn_lag_3', 'E_tn_lag_6', 'E_tn_lag_12', 'E_tn_delta_lag_1_2', 'E_tn_delta_lag_2_3', 'E_tn_delta_lag_3_6', 'E_tn_delta_lag_6_12', 'E_tn_media_3', 'E_tn_max_3', 'E_tn_min_3', 'E_tn_media_6', 'E_tn_max_6', 'E_tn_min_6', 'E_tn_media_9', 'E_tn_max_9', 'E_tn_min_9', 'E_tn_media_12', 'E_tn_max_12', 'E_tn_min_12', 'E_tn_media_18', 'E_tn_max_18', 'E_tn_min_18', 'E_tn_media_24', 'E_tn_max_24', 'E_tn_min_24', 'E_tn_media_36', 'E_tn_max_36', 'E_tn_min_36', 'E_tn_tendencia_3_6', 'E_tn_tendencia_6_9', 'E_tn_tendencia_9_12', 'E_tn_tendencia_12_18', 'E_tn_tendencia_18_24', 'E_tn_tendencia_24_36', 'ratio_E_tn_macro', 'cat1', 'cat2', 'cat3', 'brand', 'sku_size', 'descripcion', 'ratio_E_tn_cat1', 'ratio_E_tn_cat2', 'ratio_E_tn_cat3', 'ratio_E_tn_brand', 'ratio_E_tn_prod_todos_cli', 'ratio_E_tn_todos_tamanos', 'ratio_E_tn_muchos_cli', 'ratio_E_tn_complementarios']


## Split train / valid (train final 201910 -- SIN walk-forward, segun consigna)

In [8]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

m_201910 = periodo_a_meses(201910)
m_201911 = periodo_a_meses(201911)
m_201912 = periodo_a_meses(201912)

def a_pandas(tabla):
    pdf = tabla.select(features + ["clase", "TN_promedio", "clase_original"]).to_pandas()
    for c in categoricas:
        pdf[c] = pdf[c].astype("category")
    return pdf

## Entrenamiento por cluster: Optuna (tweedie, max_bin=1023) + prediccion + desescalado

In [9]:
predicciones_totales = []
predicciones_validacion = []
clusters_saltados = []

for cluster_id in sorted(df["cluster_id"].unique().to_list()):
    sub = df.filter(pl.col("cluster_id") == cluster_id)

    sub_valido = sub.filter(pl.col("clase").is_not_null())
    train = sub_valido.filter(pl.col("periodo_target_m") <= m_201910)
    valid = sub_valido.filter(
        (pl.col("periodo_target_m") >= m_201911) & (pl.col("periodo_target_m") <= m_201912)
    )

    if train.height < PARAM['min_filas_por_cluster'] or valid.height < 20:
        print(f"cluster {cluster_id}: muy poca data (train={train.height}, valid={valid.height}), se salta")
        clusters_saltados.append(cluster_id)
        continue

    train_pd = a_pandas(train)
    valid_pd = a_pandas(valid)

    # desplazo el target para que sea >=0 (requisito de tweedie), constante por cluster
    minimo_clase = min(train_pd["clase"].min(), valid_pd["clase"].min())
    constante = -minimo_clase + 1e-3 if minimo_clase < 0 else 0.0

    X_train = train_pd[features]
    y_train = train_pd["clase"] + constante

    X_valid = valid_pd[features]
    y_valid = valid_pd["clase"] + constante

    dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=categoricas,
                          params={'feature_pre_filter': False})
    dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=categoricas, reference=dtrain,
                          params={'feature_pre_filter': False})

    def objective(trial):
        params = {
            'objective': 'tweedie',
            'max_bin': PARAM['max_bin'],
            'verbosity': -1,
            'seed': PARAM['semilla'],
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 255),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 200),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        }
        modelo_trial = lgb.train(
            params, dtrain, num_boost_round=1000,
            valid_sets=[dvalid], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        return modelo_trial.best_score['valid_0']['tweedie']

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']))
    study.optimize(objective, n_trials=PARAM['n_trials'], show_progress_bar=False)

    mejores_params = dict(study.best_params)
    mejores_params.update({'objective': 'tweedie', 'max_bin': PARAM['max_bin'], 'verbosity': -1, 'seed': PARAM['semilla']})

    modelo = lgb.train(
        mejores_params, dtrain, num_boost_round=1000,
        valid_sets=[dvalid], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    futuro = sub.filter(pl.col("periodo") == PARAM['periodo_ultimo_dato'])
    futuro_pd = futuro.select(features).to_pandas()
    for c in categoricas:
        futuro_pd[c] = futuro_pd[c].astype("category")

    pred_clase_desplazada = modelo.predict(futuro_pd, num_iteration=modelo.best_iteration)
    pred_clase = pred_clase_desplazada - constante  # revierto el desplazamiento

    # desescalado: clase (delta) -> clase_original_escalada -> clase_original (tn real)
    tn0_escalado_futuro = futuro["tn0_escalado"].to_numpy()
    tn_promedio_futuro = futuro["TN_promedio"].to_numpy()

    clase_original_escalada_pred = pred_clase + tn0_escalado_futuro
    clase_original_pred = clase_original_escalada_pred * tn_promedio_futuro
    clase_original_pred = np.clip(clase_original_pred, 0, None)

    res = futuro.select(["customer_id", "product_id"]).to_pandas()
    res["tn"] = clase_original_pred
    predicciones_totales.append(res)
    valid_pred_desplazada = modelo.predict(X_valid, num_iteration=modelo.best_iteration)
    valid_pred_clase = valid_pred_desplazada - constante
    valid_clase_original_escalada_pred = valid_pred_clase + valid_pd["tn0_escalado"].to_numpy()
    valid_clase_original_pred = valid_clase_original_escalada_pred * valid_pd["TN_promedio"].to_numpy()
    valid_clase_original_pred = np.clip(valid_clase_original_pred, 0, None)

    predicciones_validacion.append(pd.DataFrame({
        "pred": valid_clase_original_pred,
        "real": valid_pd["clase_original"].to_numpy()
    }))

    print(f"cluster {cluster_id}: train={train.height} valid={valid.height} mejor_tweedie={study.best_value:.4f}")

print("\nclusters entrenados:", len(predicciones_totales), " clusters saltados:", clusters_saltados)

cluster 0: train=1580442 valid=105966 mejor_tweedie=98.6964
cluster 1: train=1565585 valid=105966 mejor_tweedie=53.9053
cluster 2: train=1557693 valid=105966 mejor_tweedie=45.0183
cluster 3: train=1540901 valid=105966 mejor_tweedie=41.9999
cluster 4: train=1512780 valid=104139 mejor_tweedie=44.2328
cluster 5: train=1537213 valid=105966 mejor_tweedie=40.5578
cluster 6: train=1447232 valid=105966 mejor_tweedie=35.0308
cluster 7: train=1368885 valid=105966 mejor_tweedie=40.0503
cluster 8: train=1215116 valid=104158 mejor_tweedie=64.3311
cluster 9: train=901514 valid=86030 mejor_tweedie=43.9066

clusters entrenados: 10  clusters saltados: []


## Combinar, SUMAR por product_id, armar submit

In [10]:
resultado_cp = pd.concat(predicciones_totales, ignore_index=True)
resultado = resultado_cp.groupby("product_id", as_index=False)["tn"].sum()

apredecir = pl.read_csv(PARAM['apredecir_path'], separator="\t").to_pandas()
submit = apredecir[["product_id"]].merge(resultado, on="product_id", how="left")
print("nulos en submit (revisar):", submit["tn"].isna().sum())
submit["tn"] = submit["tn"].fillna(0.0)

archivo_submit = os.path.join(ruta, f"{PARAM['experimento']}_submit.csv")
submit.to_csv(archivo_submit, index=False)
print(archivo_submit)
submit.head()

nulos en submit (revisar): 0
/home/ds/exp/LGB12_ESCALADO_CLUSTER/LGB12_ESCALADO_CLUSTER_submit.csv


,product_id,tn
0,20001,24769.142519
1,20002,17562.432128
2,20003,6020.119238
3,20004,4217.457090
4,20005,5457.631695


In [11]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

kaggle_submit(PARAM['kaggle_competition'], archivo_submit, f"{PARAM['experimento']} escalado+cluster bruto+tweedie")

100%|██████████| 18.6k/18.6k [00:00<00:00, 56.6kB/s]


99 submissions remaining today.
Successfully submitted to Labo III, 2026 BA

## Total Error Rate sobre el corte de validacion (WAPE, valor absoluto)
`sum(|pred-real|) / sum(real)`, calculado sobre el ultimo corte de validacion (201911-201912), a nivel producto (sumado sobre clientes), consistente con como se evalua en Kaggle.

In [ ]:
# NOTA: requiere reconstruir predicciones sobre el set de VALIDACION (no solo el futuro) para poder comparar contra el real.
# Se deja como paso siguiente si se necesita reportar el Total Error Rate en el informe -- no se calculo automaticamente
# en este notebook porque el loop de arriba solo predice el futuro (202002), no guarda las predicciones de valid por cluster.
print("Pendiente: calcular sobre predicciones de validacion guardadas, no sobre el futuro.")

In [7]:
print(df["TN_promedio"].describe())
print("TN_promedio < 0.01:", (df["TN_promedio"] < 0.01).sum(), "de", df.height)

print(df["E_tn"].describe())
print("E_tn > 100:", (df["E_tn"] > 100).sum())

print(submit["tn"].describe())

shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ value       │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 1.6648065e7 │
│ null_count ┆ 1.0         │
│ mean       ┆ 0.086279    │
│ std        ┆ 1.082736    │
│ min        ┆ 0.0         │
│ 25%        ┆ 0.0         │
│ 50%        ┆ 0.000429    │
│ 75%        ┆ 0.009289    │
│ max        ┆ 243.795543  │
└────────────┴─────────────┘
TN_promedio < 0.01: 12596928 de 16648066
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ value       │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 1.6648065e7 │
│ null_count ┆ 1.0         │
│ mean       ┆ 0.43512     │
│ std        ┆ 1.526608    │
│ min        ┆ 0.0         │
│ 25%        ┆ 0.0         │
│ 50%        ┆ 0.0         │
│ 75%        ┆ 0.0         │
│ max        ┆ 35.999725   │
└────────────┴─────────────┘
E_tn > 100: 0


NameError: name 'submit' is not defined

In [9]:
import pandas as pd
submit_guardado = pd.read_csv("/home/ds/exp/LGB12_ESCALADO_CLUSTER/LGB12_ESCALADO_CLUSTER_submit.csv")
print(submit_guardado["tn"].describe())

count      780.000000
mean       401.796680
std       1665.799975
min          0.040497
25%          8.560940
50%         39.845168
75%        182.959712
max      24769.142519
Name: tn, dtype: float64


In [10]:
top10 = submit_guardado.sort_values("tn", ascending=False).head(10)
print(top10)

print("cuantos productos con tn < 50:", (submit_guardado["tn"] < 50).sum())
print("suma total predicha:", submit_guardado["tn"].sum())

     product_id            tn
0         20001  24769.142519
31        20032  22256.070987
1         20002  17562.432128
111       20127  16209.250747
5         20006   8162.706420
27        20028   7309.344611
19        20020   6371.886117
2         20003   6020.119238
13        20014   5786.734046
4         20005   5457.631695
cuantos productos con tn < 50: 424
suma total predicha: 313401.4104645236
